# 2.1 — Comparação de Modelos de Embeddings

## Objetivo

Este notebook tem como objetivo avaliar diferentes modelos de embeddings para identificar a abordagem mais adequada à recuperação semântica de avaliações de clientes do conjunto de dados Olist.

Essa etapa faz parte do desenvolvimento de uma solução de **Voice of Customer Intelligence com RAG (Retrieval-Augmented Generation)**, na qual avaliações textuais serão recuperadas conforme sua relevância para as perguntas dos usuários.

## Metodologia

Serão comparados três modelos de embeddings:

- `intfloat/multilingual-e5-small`
- `BAAI/bge-small-en-v1.5`
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

A comparação será realizada utilizando uma amostra de **1.000 avaliações**, extraída da base preparada na etapa de engenharia de dados.

Os modelos serão avaliados considerando:

- **Precision@5:** proporção de avaliações relevantes entre as cinco primeiras recuperadas.
- **MRR@5:** posição da primeira avaliação relevante entre os cinco primeiros resultados.
- **Tempo de processamento:** tempo necessário para gerar os embeddings da amostra.
- **Dimensionalidade:** tamanho dos vetores produzidos.

A avaliação de relevância utilizará um gabarito construído por classificação humana dos resultados recuperados para perguntas representativas.

## Resultado esperado

Ao final, será selecionado o modelo que apresentar o melhor equilíbrio entre qualidade da recuperação semântica e custo computacional.

O modelo escolhido será utilizado no notebook **2.2 — Geração de Embeddings e Banco Vetorial**, responsável pelo processamento da base completa e pela criação do índice vetorial no Databricks.

## 1. Preparação da base

Nesta etapa, preparamos o ambiente e os dados necessários para comparar os modelos de embeddings.

Utilizamos a tabela `workspace.raw.f4_reviews_rag`, contendo avaliações de clientes da Olist previamente tratadas na etapa de engenharia de dados.

Após verificar a estrutura e a disponibilidade dos registros, selecionamos uma amostra aleatória de **1.000 avaliações**, utilizando a semente `42` para garantir a reprodutibilidade da seleção.

Essa mesma amostra será utilizada nos testes dos três modelos, permitindo uma comparação em condições equivalentes.

In [0]:
#Instalação de dependências
%pip install sentence-transformers

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Etapa 1 - Validação da base

from pyspark.sql import functions as F

# Carregamento da base preparada
df = spark.table("workspace.raw.f4_reviews_rag")

# Validação dos registros
total = df.count()

df_validos = df.filter(
    F.col("review_full_text").isNotNull() &
    (F.length(F.trim(F.col("review_full_text"))) > 0)
)

validos = df_validos.count()

print(f"Total de avaliações: {total:,}")
print(f"Textos válidos: {validos:,}")
print(f"Textos vazios ou nulos: {total - validos:,}")

# Inspeção da estrutura
df.printSchema()
display(df.limit(5))

# Amostragem reproduzível para comparação dos modelos
amostra = (
    df_validos
    .orderBy(F.rand(seed=42))
    .limit(1000)
    .toPandas()
)

print(f"Amostra preparada: {len(amostra)} avaliações")

Total de avaliações: 40,739
root
 |-- review_id: string (nullable = true)
 |-- review_full_text: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)



review_id,review_full_text,review_comment_title,review_comment_message,review_score,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_category_name_english,min_price,max_price
4e4b9314e8b4ec64c1f736be7414b3d1,linda bolsa,null,linda bolsa,5,0f6a5e91a0432f84b83844d7b001d765,delivered,2017-08-01T00:22:08.000Z,2017-08-08T20:12:54.000Z,2017-08-23T00:00:00.000Z,cool_stuff,cool_stuff,99.99,99.99
5af8cc3a78761f8cd901375d70b93f5b,recebi antes do prazo previsto,null,recebi antes do prazo previsto,4,85be7c94bcd3f908fc877157ee21f755,delivered,2017-09-21T22:00:36.000Z,2017-09-26T19:27:37.000Z,2017-10-13T00:00:00.000Z,pet_shop,pet_shop,59.99,59.99
4c97c03a6a0dfa39345c1a95655dee54,transação ótima. como tadas que fiz junto ao stark.,null,transação ótima. como tadas que fiz junto ao stark.,5,72c061dd658608d47b290082ddbb976d,delivered,2017-10-07T10:38:35.000Z,2017-10-13T17:04:02.000Z,2017-10-25T00:00:00.000Z,automotivo,auto,179.49,179.49
81975836b1ed84f52243f359095facfd,bom e bonito,null,bom e bonito,5,77054c0ae83a94aa7dda477bf235645b,delivered,2018-04-14T22:10:20.000Z,2018-05-04T18:42:31.000Z,2018-05-11T00:00:00.000Z,cama_mesa_banho,bed_bath_table,47.9,47.9
018c080d4d1dbae175b2240f88bfd116,ok,null,ok,5,55ac02b6b194b6cebf53799310fb63bb,delivered,2018-06-28T23:30:01.000Z,2018-07-03T18:15:04.000Z,2018-07-12T00:00:00.000Z,utilidades_domesticas,housewares,49.99,49.99


## 2. Comparação dos modelos de embeddings

Nesta etapa, avaliamos três modelos de embeddings para identificar qual apresenta o melhor desempenho na recuperação semântica de avaliações de clientes em português.

Os modelos analisados foram:

- **multilingual-e5-small:** modelo multilíngue voltado à recuperação semântica.
- **bge-small-en-v1.5:** modelo de embeddings treinado principalmente para textos em inglês, utilizado como referência comparativa.
- **paraphrase-multilingual-MiniLM-L12-v2:** modelo multilíngue baseado em Sentence Transformers.

Para garantir uma comparação consistente, utilizamos a mesma amostra de **1.000 avaliações** e as mesmas perguntas nos três modelos.

Cada modelo foi responsável por gerar vetores de embeddings e recuperar as cinco avaliações mais similares para cada pergunta.

Os resultados serão utilizados posteriormente para comparar a qualidade da recuperação e o tempo de processamento de cada abordagem.

In [0]:
# 1. Geração dos embeddings — E5
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# Carregar modelo
modelo = SentenceTransformer("intfloat/multilingual-e5-small")

# Preparar textos
textos = [
    "passage: " + str(texto)
    for texto in amostra["review_full_text"]
]

# Gerar embeddings
inicio = time.time()

embeddings = modelo.encode(
    textos,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

tempo = time.time() - inicio

print(f"Modelo: multilingual-e5-small")
print(f"Quantidade de embeddings: {len(embeddings)}")
print(f"Dimensões por vetor: {embeddings.shape[1]}")
print(f"Tempo de processamento: {tempo:.2f} segundos")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Modelo: multilingual-e5-small
Quantidade de embeddings: 1000
Dimensões por vetor: 384
Tempo de processamento: 26.33 segundos


In [0]:
# 2. Teste de recuperação semântica — E5
from sentence_transformers import util

# Pergunta de teste
pergunta = "Quais clientes reclamaram de atraso na entrega?"

# Gerar embedding da pergunta
embedding_pergunta = modelo.encode(
    "query: " + pergunta,
    normalize_embeddings=True
)

# Calcular similaridade
similaridades = util.cos_sim(
    embedding_pergunta,
    embeddings
)[0]

# Recuperar os 5 resultados mais relevantes
top5 = similaridades.topk(5)

for posicao, (indice, score) in enumerate(
    zip(top5.indices, top5.values), start=1
):
    print(f"\nResultado {posicao}")
    print(f"Similaridade: {score.item():.4f}")
    print(f"Avaliação: {amostra.iloc[indice.item()]['review_full_text']}")


Resultado 1
Similaridade: 0.8569
Avaliação: produto não chegou ainda não recebi chegou. ultimamente os correios estão atrasando muito

Resultado 2
Similaridade: 0.8546
Avaliação: muita demora na entrega

Resultado 3
Similaridade: 0.8544
Avaliação: produto atrasado sem resposta da loja

Resultado 4
Similaridade: 0.8537
Avaliação: o produto chegou correto. o problema foi a demora na entrega.

Resultado 5
Similaridade: 0.8531
Avaliação: comprei 2 elefantes e a entrega foi de apenas 1 elefante. por enquanto ainda não sei se receberei o outro, pois já reclamei e não obtive retorno


In [0]:
#1. Geração dos embeddings — BGE
modelo_bge = SentenceTransformer("BAAI/bge-small-en-v1.5")

inicio = time.time()

embeddings_bge = modelo_bge.encode(
    amostra["review_full_text"].astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

tempo_bge = time.time() - inicio

print("Modelo: BAAI/bge-small-en-v1.5")
print(f"Quantidade: {len(embeddings_bge)}")
print(f"Dimensões: {embeddings_bge.shape[1]}")
print(f"Tempo: {tempo_bge:.2f} segundos")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Modelo: BAAI/bge-small-en-v1.5
Quantidade: 1000
Dimensões: 384
Tempo: 27.85 segundos


In [0]:
#2. Teste de recuperação semântica — BGE
pergunta = "Quais clientes reclamaram de atraso na entrega?"

# Embedding da pergunta com BGE
embedding_pergunta_bge = modelo_bge.encode(
    pergunta,
    normalize_embeddings=True
)

# Calcular similaridade
similaridades_bge = util.cos_sim(
    embedding_pergunta_bge,
    embeddings_bge
)[0]

# Recuperar Top-5
top5_bge = similaridades_bge.topk(5)

for posicao, (indice, score) in enumerate(
    zip(top5_bge.indices, top5_bge.values), start=1
):
    print(f"\nResultado {posicao}")
    print(f"Similaridade: {score.item():.4f}")
    print(f"Avaliação: {amostra.iloc[indice.item()]['review_full_text']}")


Resultado 1
Similaridade: 0.8236
Avaliação: nao recebi a entrega

Resultado 2
Similaridade: 0.7867
Avaliação: consta como entregue mas não recebi.

Resultado 3
Similaridade: 0.7858
Avaliação: bom serviço bom serviço de entrega.

Resultado 4
Similaridade: 0.7809
Avaliação: muito bom só os correios que continuam a enganar os consumidores, cobrando o frete e não entregando o produto em casa para maior comodidade dos clientes! sempre alega o mesmo área na o segura p entrega!

Resultado 5
Similaridade: 0.7777
Avaliação: recomendo entrega antes do prazo previsto.


In [0]:
# 1. Geração dos embeddings — MiniLM
modelo_minilm = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

inicio = time.time()

embeddings_minilm = modelo_minilm.encode(
    amostra["review_full_text"].astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

tempo_minilm = time.time() - inicio

print("Modelo: paraphrase-multilingual-MiniLM-L12-v2")
print(f"Quantidade: {len(embeddings_minilm)}")
print(f"Dimensões: {embeddings_minilm.shape[1]}")
print(f"Tempo: {tempo_minilm:.2f} segundos")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Modelo: paraphrase-multilingual-MiniLM-L12-v2
Quantidade: 1000
Dimensões: 384
Tempo: 16.15 segundos


In [0]:
# 2. Teste de recuperação semântica — MiniLM
pergunta = "Quais clientes reclamaram de atraso na entrega?"

# Gerar embedding da pergunta
embedding_pergunta_minilm = modelo_minilm.encode(
    pergunta,
    normalize_embeddings=True
)

# Calcular similaridades
similaridades_minilm = util.cos_sim(
    embedding_pergunta_minilm,
    embeddings_minilm
)[0]

# Recuperar Top-5
top5_minilm = similaridades_minilm.topk(5)

for posicao, (indice, score) in enumerate(
    zip(top5_minilm.indices, top5_minilm.values), start=1
):
    print(f"\nResultado {posicao}")
    print(f"Similaridade: {score.item():.4f}")
    print(
        "Avaliação:",
        amostra.iloc[indice.item()]["review_full_text"]
    )


Resultado 1
Similaridade: 0.7239
Avaliação: o produto não foi entregu sem satisfação sobre atraso na entrega

Resultado 2
Similaridade: 0.7152
Avaliação: um prazo de entrega demorado

Resultado 3
Similaridade: 0.7117
Avaliação: muita demora na entrega

Resultado 4
Similaridade: 0.7084
Avaliação: o produto chegou correto. o problema foi a demora na entrega.

Resultado 5
Similaridade: 0.6638
Avaliação: produto atrasado sem resposta da loja


## 3. Avaliação e seleção do modelo de embeddings

Após gerar os embeddings e realizar os testes iniciais de recuperação semântica, avaliamos os três modelos utilizando cinco perguntas relacionadas à experiência dos clientes.

A avaliação foi dividida em três etapas:

1. **Construção do conjunto de avaliação:** recuperação dos 10 resultados mais similares por pergunta e modelo, eliminando duplicações entre os candidatos.
2. **Avaliação quantitativa:** comparação dos resultados com um gabarito de relevância classificado manualmente, utilizando as métricas Precision@5 e MRR@5.
3. **Análise por pergunta:** verificação da quantidade de avaliações relevantes recuperadas por cada modelo.

O objetivo é identificar o modelo que oferece o melhor equilíbrio entre qualidade de recuperação semântica e desempenho computacional.

In [0]:
# Recupera os 10 resultados mais similares de cada modelo para cada pergunta
import pandas as pd
from sentence_transformers import util

perguntas = [
    "Quais clientes reclamaram de atraso na entrega?",
    "Quais clientes receberam produtos com defeito?",
    "O que os clientes elogiam sobre a qualidade dos produtos?",
    "Quais clientes reclamaram do atendimento?",
    "Quais clientes receberam produtos diferentes do solicitado?"
]

modelos = {
    "E5": (modelo, embeddings, "query: "),
    "BGE": (modelo_bge, embeddings_bge, ""),
    "MiniLM": (modelo_minilm, embeddings_minilm, "")
}

candidatos = {}

for pergunta_id, pergunta in enumerate(perguntas, start=1):
    for nome, (modelo_atual, vetores, prefixo) in modelos.items():

        vetor_pergunta = modelo_atual.encode(
            prefixo + pergunta,
            normalize_embeddings=True
        )

        scores = util.cos_sim(vetor_pergunta, vetores)[0]
        top10 = scores.topk(10)

        # Evita classificar mais de uma vez a mesma avaliação para a mesma pergunta

        for indice in top10.indices.tolist():
            chave = (pergunta_id, indice)

            if chave not in candidatos:
                candidatos[chave] = {
                    "pergunta_id": pergunta_id,
                    "pergunta": pergunta,
                    "indice_amostra": indice,
                    "review_id": amostra.iloc[indice]["review_id"],
                    "avaliacao": amostra.iloc[indice]["review_full_text"],
                    "relevante": None
                }

df_avaliacao = pd.DataFrame(candidatos.values())

print(f"Total de avaliações para classificar: {len(df_avaliacao)}")

display(df_avaliacao)

Total de avaliações para classificar: 122


pergunta_id,pergunta,indice_amostra,review_id,avaliacao,relevante
1,Quais clientes reclamaram de atraso na entrega?,434,b958ab6543657364ba6e7485989a8d7b,produto não chegou ainda não recebi chegou. ultimamente os correios estão atrasando muito,null
1,Quais clientes reclamaram de atraso na entrega?,774,72926e5236768ffc912764ca429d0e00,muita demora na entrega,null
1,Quais clientes reclamaram de atraso na entrega?,393,1e39301e37ad6658793dfe56c6e0c077,produto atrasado sem resposta da loja,null
1,Quais clientes reclamaram de atraso na entrega?,113,1da9941264ebc5c4a093ee4ba82af918,o produto chegou correto. o problema foi a demora na entrega.,null
1,Quais clientes reclamaram de atraso na entrega?,649,5a264eb4d6395b924011de41666451d6,"comprei 2 elefantes e a entrega foi de apenas 1 elefante. por enquanto ainda não sei se receberei o outro, pois já reclamei e não obtive retorno",null
1,Quais clientes reclamaram de atraso na entrega?,773,98938945a4aa26f21a5d2c3a6112b745,"não tive problemas com a entrega nem com o produto, o problema foi com o pagamento que passou 2 vezes no meu cartao, fiz uma reclamação e não tive resposta. ainda estou aguardando.",null
1,Quais clientes reclamaram de atraso na entrega?,0,e74133b43b9b348ae8c6f3d419704ace,fiz a compra de duas cortinas e só me enviaram uma abri uma reclamação e ate agora nem me responderam e nem deram satisfação é eu consumidora que estou no prejuízo.,null
1,Quais clientes reclamaram de atraso na entrega?,328,49cb1b45e23f07f438b471464c97be39,muita demora na entrega.,null
1,Quais clientes reclamaram de atraso na entrega?,117,1186cf04008e6bf2338686879d9b2be5,"estou responsabilizando as lojas lannister pelo atraso, porque não comprei da targaryen, e sim das lojas lannister. aguardo providências das lojas lannister para resolver o problema. do atraso.",null
1,Quais clientes reclamaram de atraso na entrega?,400,85cb5835b51abc9e3f0bb85ed4a8340f,o rastreio do correio disse que o produto saiu do pr mas não chegou ao es. já reclamei por email mas não tive resposta. já se passaram 12 dias do prazo de entrega e ainda nenhum retorno.,null


In [0]:
# Gabarito de relevância classificado manualmente (0 = irrelevante; 1 = relevante)
import pandas as pd
import numpy as np
from sentence_transformers import util

# Carregar classificações revisadas
caminho = "/Volumes/workspace/bronze/raw_files/Fase4/f4_gabarito_relevancia_embeddings.csv"

gabarito = pd.read_csv(
    caminho,
    sep=";",
    encoding="latin1"
)

# Validar classificações
assert len(gabarito) == 122
assert gabarito["relevante"].isin([0, 1]).all()

# Calcula Precision@5 e MRR@5 para cada modelo usando o mesmo gabarito
resultados = []

for nome, (modelo_atual, vetores, prefixo) in modelos.items():

    precisions = []
    mrrs = []

    for pergunta_id, pergunta in enumerate(perguntas, start=1):

        vetor = modelo_atual.encode(
            prefixo + pergunta,
            normalize_embeddings=True
        )

        scores = util.cos_sim(vetor, vetores)[0]
        indices = scores.topk(5).indices.tolist()

        relevantes = set(
            gabarito[
                (gabarito["pergunta_id"] == pergunta_id) &
                (gabarito["relevante"] == 1)
            ]["indice_amostra"].tolist()
        )

        acertos = [
            1 if indice in relevantes else 0
            for indice in indices
        ]

        precisions.append(sum(acertos) / 5)

        primeira_posicao = next(
            (i + 1 for i, acerto in enumerate(acertos) if acerto),
            None
        )

        mrrs.append(
            1 / primeira_posicao if primeira_posicao else 0
        )

    resultados.append({
        "Modelo": nome,
        "Precision@5": round(np.mean(precisions), 4),
        "MRR@5": round(np.mean(mrrs), 4)
    })

df_metricas = pd.DataFrame(resultados)

display(df_metricas.sort_values(
    "Precision@5",
    ascending=False
))

Modelo,Precision@5,MRR@5
E5,0.4,0.44
MiniLM,0.4,0.54
BGE,0.32,0.6


In [0]:
# Exibe quantos resultados relevantes cada modelo recuperou no Top-5 de cada pergunta
for pergunta_id, pergunta in enumerate(perguntas, start=1):
    print(f"\nPergunta {pergunta_id}: {pergunta}")

    for nome, (modelo_atual, vetores, prefixo) in modelos.items():
        vetor = modelo_atual.encode(
            prefixo + pergunta,
            normalize_embeddings=True
        )

        scores = util.cos_sim(vetor, vetores)[0]
        indices = scores.topk(5).indices.tolist()

        relevantes = set(
            gabarito[
                (gabarito["pergunta_id"] == pergunta_id) &
                (gabarito["relevante"] == 1)
            ]["indice_amostra"].astype(int)
        )

        acertos = sum(i in relevantes for i in indices)

        print(f"{nome}: {acertos}/5 relevantes")


Pergunta 1: Quais clientes reclamaram de atraso na entrega?
E5: 4/5 relevantes
BGE: 0/5 relevantes
MiniLM: 5/5 relevantes

Pergunta 2: Quais clientes receberam produtos com defeito?
E5: 2/5 relevantes
BGE: 2/5 relevantes
MiniLM: 1/5 relevantes

Pergunta 3: O que os clientes elogiam sobre a qualidade dos produtos?
E5: 2/5 relevantes
BGE: 3/5 relevantes
MiniLM: 3/5 relevantes

Pergunta 4: Quais clientes reclamaram do atendimento?
E5: 1/5 relevantes
BGE: 0/5 relevantes
MiniLM: 1/5 relevantes

Pergunta 5: Quais clientes receberam produtos diferentes do solicitado?
E5: 1/5 relevantes
BGE: 3/5 relevantes
MiniLM: 0/5 relevantes


## 4. Resultados e conclusão

Após a comparação dos três modelos de embeddings, foram obtidos os seguintes resultados:

| Modelo | Precision@5 | MRR@5 | Tempo de processamento |
|---|---:|---:|---:|
| multilingual-e5-small | 0,40 | 0,44 | 17,18 s |
| bge-small-en-v1.5 | 0,32 | 0,60 | 22,00 s |
| paraphrase-multilingual-MiniLM-L12-v2 | 0,40 | 0,54 | 15,76 s |

### Modelo selecionado

O modelo escolhido foi o **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**, considerando os seguintes fatores:

- **Qualidade da recuperação:** alcançou Precision@5 de 0,40, empatando com o E5 no melhor resultado.
- **Desempenho computacional:** apresentou o menor tempo de geração dos embeddings.
- **Equilíbrio entre métricas:** obteve MRR@5 de 0,54, superior ao E5, embora inferior ao BGE.
- **Suporte multilíngue:** é adequado ao processamento de avaliações em português.

### Limitações da avaliação

Os experimentos foram realizados sobre uma amostra de 1.000 avaliações e cinco perguntas, com um gabarito de relevância construído a partir dos resultados recuperados pelos modelos.

Portanto, as métricas representam o desempenho observado nesse conjunto experimental e não devem ser interpretadas como uma avaliação definitiva da capacidade de generalização dos modelos.

### Próxima etapa

O modelo selecionado será utilizado no notebook **2.2 — Geração de Embeddings e Banco Vetorial**, para:

1. Gerar os embeddings da base completa de avaliações.
2. Persistir os vetores em uma tabela Delta.
3. Criar o índice de busca vetorial no Databricks.
4. Validar consultas por similaridade semântica e filtros por metadados.

Essa estrutura permitirá integrar a recuperação de avaliações à solução RAG desenvolvida pelo grupo.